<a href="https://colab.research.google.com/github/mango208014-ai/Green-GML-Profiler/blob/main/metrics_mlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import time
import pandas as pd
import psutil
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim


# 1. Reuse your HardwareProfiler class from Week 2/3
class HardwareProfiler:

  def __init__(self):
    self.process = psutil.Process(os.getpid())
    self.start_time = None
    self.peak_ram = 0

  def start(self):
    self.peak_ram = self.process.memory_info().rss
    self.start_time = time.time()

  def stop(self):
    end_time = time.time()
    current_ram = self.process.memory_info().rss
    if current_ram > self.peak_ram:
      self.peak_ram = current_ram

    latency = end_time - self.start_time
    peak_ram_mb = self.peak_ram / (1024 * 1024)
    power_kw = 0.015
    carbon_gco2e = (latency / 3600) * power_kw * 400
    return {
        "latency_seconds": round(latency, 4),
        "peak_ram_mb": round(peak_ram_mb, 2),
        "carbon_gco2e": round(carbon_gco2e, 6),
    }


# 2. Data Pipeline (Identical to Week 1 & 3)
wine = load_wine()
X = StandardScaler().fit_transform(wine.data)
X_pca = PCA(n_components=4).fit_transform(X)
y = wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.2, random_state=42
)

# Convert arrays to PyTorch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)


# 3. Define Classical MLP (4 inputs -> 1 hidden layer of 4 nodes -> 3 wine classes)
class ClassicalMLP(nn.Module):

  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(4, 4),  # 4 PCA inputs -> 1 hidden layer (4 units)
        nn.ReLU(),
        nn.Linear(4, 3),  # 3 output classes for UCI Wine
    )

  def forward(self, x):
    return self.net(x)


# 4. Execute 20 profiling trials (20 epochs each)
profiler = HardwareProfiler()
results = []
num_trials = 20
epochs = 20

print("Running 20 profiling trials for Classical MLP (20 epochs each)...")

for trial in range(num_trials):
  # Re-initialize model & optimizer fresh for each trial
  model = ClassicalMLP()
  criterion = nn.CrossEntropyLoss()
  optimizer = optim.Adam(model.parameters(), lr=0.01)

  profiler.start()

  # 20-epoch training loop
  for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()

  # Evaluation
  with torch.no_grad():
    preds = torch.argmax(model(X_test_t), dim=1).numpy()
    acc = accuracy_score(y_test, preds)

  metrics = profiler.stop()

  results.append({
      "model": "Classical_MLP",
      "trial": trial + 1,
      "accuracy": round(acc, 4),
      "latency_seconds": metrics["latency_seconds"],
      "peak_ram_mb": metrics["peak_ram_mb"],
      "carbon_gco2e": metrics["carbon_gco2e"],
  })

# 5. Save outcomes to CSV
df_mlp = pd.DataFrame(results)
df_mlp.to_csv("metrics_mlp.csv", index=False)
print("\nWeek 4 profiling complete! Summary averages:")
print(
    df_mlp[["accuracy", "latency_seconds", "peak_ram_mb", "carbon_gco2e"]].mean()
)

Running 20 profiling trials for Classical MLP (20 epochs each)...

Week 4 profiling complete! Summary averages:
accuracy             0.709720
latency_seconds      0.021870
peak_ram_mb        526.930000
carbon_gco2e         0.000037
dtype: float64
